# Heat and transport in 1D

Finite differences turn a differential equation into a sparse linear system,
and the equation decides which solver may be used. Steady heat flow through a
rod whose conductivity varies gives a symmetric positive definite matrix, so
the conjugate gradient method applies, and a preconditioner can make up for
the uneven conductivity. Add transport, a flow carrying the heat along the
rod, and the matrix is no longer symmetric. CG then fails, and BiCGSTAB is
the method to use.

Both problems have exact solutions, so the notebook measures the
discretization's error as well as the solvers' iteration counts.

**Uses:** `mtl5.sparse.coo_matrix`, `mtl5.sparse.cg`, `mtl5.sparse.bicgstab`,
`mtl5.sparse.diagonal`, `full_output=` and `callback=` on the solvers.

**From C++:** [`phase03_iterative_solvers/heat_equation_1d.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/phase03_iterative_solvers/heat_equation_1d.cpp), [`phase03_iterative_solvers/convection_diffusion.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/phase03_iterative_solvers/convection_diffusion.cpp).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import mtl5.sparse as ms

# Chart styling: recessive axes and grid, ink for text, validated categorical hues.
SURFACE = "#fcfcfb"
INK, INK_2 = "#0b0b0b", "#52514e"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "text.color": INK,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "font.size": 10,
        "figure.dpi": 120,
    }
)


def tridiagonal(lower, diag, upper):
    # A sparse n x n matrix with lower[i] at (i, i - 1), diag[i] at (i, i)
    # and upper[i] at (i, i + 1); lower[0] and upper[-1] are unused.
    n = len(diag)
    i = np.arange(n)
    rows = np.concatenate([i, i[1:], i[:-1]])
    cols = np.concatenate([i, i[1:] - 1, i[:-1] + 1])
    vals = np.concatenate([diag, lower[1:], upper[:-1]])
    return ms.coo_matrix(rows, cols, vals.astype(np.float64), (n, n)).tocsr()

## Heat flow with variable conductivity

The temperature u in a rod on [0, 1], held at 0 at both ends and heated
uniformly, satisfies −(k(x) u′)′ = 1. The conductivity k(x) = 1 + 99x rises
from 1 to 100 along the rod. The standard discretization at n interior points,
with h = 1 / (n + 1) and k taken at the midpoints between them, is

(−k₋ uᵢ₋₁ + (k₋ + k₊) uᵢ − k₊ uᵢ₊₁) / h² = 1,

where k₋ = k(xᵢ − h/2) and k₊ = k(xᵢ + h/2). The matrix is symmetric because
the coefficient between points i and i + 1 is the same k₊ seen from either
side. It is positive definite because k > 0.

The equation integrates in closed form: k u′ = C − x, so

u(x) = ln(1 + 99x) / (99 ln 100) − x / 99.

In [ ]:
def heat_system(n):
    h = 1.0 / (n + 1)
    x = h * np.arange(1, n + 1)
    k_minus, k_plus = 1.0 + 99.0 * (x - h / 2), 1.0 + 99.0 * (x + h / 2)
    A = tridiagonal(-k_minus / h**2, (k_minus + k_plus) / h**2, -k_plus / h**2)
    return x, A, np.ones(n)


def u_exact(x):
    return np.log1p(99.0 * x) / (99.0 * np.log(100.0)) - x / 99.0


n = 100
x, A, b = heat_system(n)
indptr, _, values = A.to_csr_arrays()
diagonal = np.array([values[indptr[i] : indptr[i + 1]].max() for i in range(n)])
ratio = diagonal[-1] / diagonal[0]
print(f"diagonal from {diagonal[0]:.0f} to {diagonal[-1]:.0f}, a ratio of {ratio:.0f}")

## CG, and CG with a Jacobi preconditioner

The diagonal varies by a factor of 50 along the rod, so the rows of A are on
very different scales. The Jacobi preconditioner M = diag(A) rescales every row to
a unit diagonal, which is what CG's convergence depends on. With
`full_output=True` each solve also returns its residual history.

In [ ]:
u_plain, info_plain, plain = ms.cg(A, b, rtol=1e-10, maxiter=1000, full_output=True)
u_jac, info_jac, jacobi = ms.cg(A, b, M=ms.diagonal(A), rtol=1e-10, maxiter=1000, full_output=True)
print(f"CG:          {plain.iterations} iterations")
print(f"CG + Jacobi: {jacobi.iterations} iterations")
print(f"largest difference between the two solutions: {np.abs(u_plain - u_jac).max():.1e}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.0, 3.6))
x_fine = np.linspace(0.0, 1.0, 400)
ax1.plot(x_fine, u_exact(x_fine), color=INK_2, linewidth=1, label="exact")
ax1.plot(x[::4], u_jac[::4], "o", color=SERIES[0], markersize=3, label="computed (every 4th point)")
ax1.set_xlabel("x")
ax1.set_ylabel("u(x)")
ax1.set_title("Temperature along the rod", fontsize=10)
ax1.legend(frameon=False, fontsize=8)
ax1.grid(True)
for color, (label, report) in zip(SERIES, [("CG", plain), ("CG + Jacobi", jacobi)]):
    ax2.semilogy(report.history / report.history[0], color=color, label=label)
ax2.set_xlabel("iteration")
ax2.set_ylabel("residual / ||b||")
ax2.set_title("Convergence", fontsize=10)
ax2.legend(frameon=False, fontsize=8)
ax2.grid(True)
plt.tight_layout()
plt.show()

assert info_plain == 0 and info_jac == 0
assert jacobi.iterations < plain.iterations
assert np.allclose(u_plain, u_jac, rtol=0, atol=1e-12)  # the same answer

Preconditioning changes how fast CG gets there, not where it ends up. In exact
arithmetic CG finishes in at most n steps, one per eigenvalue. With the
Jacobi preconditioner it finishes at about n. Unpreconditioned it needs
almost half as many again, because on the badly scaled matrix rounding erodes
the orthogonality of its search directions.

In [ ]:
assert jacobi.iterations <= n + 5
assert plain.iterations >= 1.2 * n
print(f"n = {n}: CG + Jacobi {jacobi.iterations}, CG {plain.iterations}")

## How accurate is the discretization?

The solver error is now far below the discretization error, so the difference
from the exact solution measures the finite differences themselves. The
scheme is second order: halving h should divide the error by 4. That rate is
reached once the grid resolves the steep part of the solution near x = 0.

In [ ]:
sizes = np.array([25, 50, 100, 200, 400])
errors = []
for m in sizes:
    xm, Am, bm = heat_system(m)
    um, info, _ = ms.cg(Am, bm, M=ms.diagonal(Am), rtol=1e-10, maxiter=5 * m, full_output=True)
    assert info == 0
    errors.append(np.abs(um - u_exact(xm)).max())
errors = np.array(errors)
h = 1.0 / (sizes + 1)

fig, ax = plt.subplots(figsize=(5.2, 3.4))
ax.loglog(h, errors, "o-", color=SERIES[0], label="max error")
ax.loglog(h, errors[-1] * (h / h[-1]) ** 2, ":", color=INK_2, label="slope 2")
ax.set_xlabel("h")
ax.set_ylabel("max |u_h − u|")
ax.set_title("Second-order convergence", fontsize=10)
ax.legend(frameon=False)
ax.grid(True)
plt.tight_layout()
plt.show()

ratios = errors[:-1] / errors[1:]
print("error ratio per halving of h:", np.round(ratios, 2))
assert (ratios > 2.0).all() and ratios[-1] > 3.5

## Transport: a matrix that isn't symmetric

Now let the heat be carried along the rod at unit speed while it diffuses:
−ε u″ + u′ = 1, with u = 0 at both ends. Central differences give

(−ε (uᵢ₋₁ − 2uᵢ + uᵢ₊₁) / h²) + (uᵢ₊₁ − uᵢ₋₁) / (2h) = 1.

The first-derivative term enters the two off-diagonals with opposite signs, so
A is no longer symmetric. The smaller ε, the stronger the asymmetry. The cell
Péclet number Pe = h / (2ε) measures it. The exact solution is

u(x) = x − (e^(x/ε) − 1) / (e^(1/ε) − 1),

which rises like x and drops to 0 in a boundary layer of width about ε at
x = 1.

In [ ]:
def transport_system(n, eps):
    h = 1.0 / (n + 1)
    x = h * np.arange(1, n + 1)
    lower = np.full(n, -eps / h**2 - 1.0 / (2 * h))
    upper = np.full(n, -eps / h**2 + 1.0 / (2 * h))
    return x, tridiagonal(lower, np.full(n, 2 * eps / h**2), upper), np.ones(n)


def u_transport(x, eps):
    # x - expm1(x/eps) / expm1(1/eps), written to avoid overflow for small eps
    return x - np.exp((x - 1.0) / eps) * (-np.expm1(-x / eps)) / (-np.expm1(-1.0 / eps))


n_t = 30  # the C++ example's grid
regimes = {}
for eps in (1.0, 0.1, 0.01):
    xt, At, bt = transport_system(n_t, eps)
    u_cg, info_cg, cg_report = ms.cg(At, bt, rtol=1e-8, maxiter=500, full_output=True)
    u_bi, info_bi, bi_report = ms.bicgstab(At, bt, rtol=1e-8, maxiter=500, full_output=True)
    regimes[eps] = (xt, u_bi, cg_report, bi_report, info_cg, info_bi)
    pe = (1.0 / (n_t + 1)) / (2 * eps)
    print(
        f"eps = {eps:<5} Pe = {pe:4.2f} | CG: info {info_cg}, relative residual "
        f"{cg_report.relative_residual:.1e} | BiCGSTAB: info {info_bi}, "
        f"{bi_report.iterations} iterations, relative residual {bi_report.relative_residual:.1e}"
    )

BiCGSTAB converges in every regime. CG never does, not even for ε = 1, where
the matrix is nearly symmetric and CG stops just short of the tolerance after
its 500 iterations. Its derivation assumes A = Aᵀ, and without that
assumption its residual has no reason to decrease. In the convection-dominated
regimes it grows instead.

BiCGSTAB's residual isn't monotone either: at ε = 0.01 it climbs by six
orders of magnitude before it converges. Smoothing out such excursions is
what BiCGSTAB(ℓ) is for, in the [Krylov solvers](krylov_comparison.ipynb)
notebook.

In [ ]:
for eps, (_, _, cg_report, bi_report, info_cg, info_bi) in regimes.items():
    assert info_bi == 0 and bi_report.relative_residual <= 1e-8, eps
for eps in (0.1, 0.01):
    _, _, cg_report, _, info_cg, _ = regimes[eps]
    assert info_cg != 0 and cg_report.relative_residual > 1.0, eps

fig, ax = plt.subplots(figsize=(5.8, 3.4))
_, _, cg_report, bi_report, _, _ = regimes[0.01]
ax.semilogy(cg_report.history / cg_report.history[0], color=SERIES[1], label="CG")
ax.semilogy(bi_report.history / bi_report.history[0], color=SERIES[0], label="BiCGSTAB")
ax.set_xlabel("iteration")
ax.set_ylabel("residual / ||b||")
ax.set_title("ε = 0.01: CG's residual grows, BiCGSTAB's converges", fontsize=10)
ax.legend(frameon=False)
ax.grid(True)
plt.tight_layout()
plt.show()

## When the grid is too coarse for the flow

A converged solver gives the discrete solution, which is only as good as the
discretization. Central differences are accurate while Pe < 1. At ε = 0.01 on
30 points, Pe = 1.6: the grid can't resolve the boundary layer, and the
discrete solution overshoots and wiggles near x = 1, a numerical artifact the
exact solution doesn't have.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9.6, 3.0), sharey=True)
max_errors = {}
for ax, (eps, (xt, u_bi, *_)) in zip(axes, regimes.items()):
    xf = np.linspace(0.0, 1.0, 600)
    ax.plot(xf, u_transport(xf, eps), color=INK_2, linewidth=1, label="exact")
    ax.plot(
        np.r_[0, xt, 1], np.r_[0, u_bi, 0], "o-", color=SERIES[0], markersize=3, label="computed"
    )
    pe = (1.0 / (n_t + 1)) / (2 * eps)
    ax.set_title(f"ε = {eps}, Pe = {pe:.2f}", fontsize=10)
    ax.set_xlabel("x")
    ax.grid(True)
    max_errors[eps] = np.abs(u_bi - u_transport(xt, eps)).max()
axes[0].set_ylabel("u(x)")
axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

for eps, err in max_errors.items():
    print(f"eps = {eps:<5} max error of the discrete solution {err:.1e}")
assert max_errors[0.01] > 10 * max_errors[0.1]
assert regimes[0.01][1].max() > 1.0  # the overshoot: the exact solution stays below 1

## Watching a solve

The C++ example shows MTL5's three iteration controllers: `basic_iteration`
is silent, `cyclic_iteration` prints every few steps and `noisy_iteration`
prints every step. From Python, a callback plays that role. It receives the
iteration count and the residual norm, and can print, record or plot them.

In [ ]:
_, A_mid, b_mid = transport_system(n_t, 0.1)


def every_5(k, rnorm):
    if k % 5 == 0:
        print(f"iteration {k:3d}   residual {rnorm:.3e}")


_, info, report = ms.bicgstab(A_mid, b_mid, rtol=1e-8, callback=every_5, full_output=True)
print(
    f"converged in {report.iterations} iterations, relative residual {report.relative_residual:.1e}"
)
assert info == 0

## From C++

| MTL5 (C++) | mtl5 (Python) |
| --- | --- |
| `mat::dense2D<double> A(n, n)`, filled entry by entry | `ms.coo_matrix(rows, cols, values, shape).tocsr()`, sparse |
| `itl::cg(A, x, b, pc, iter)` | `x, info = ms.cg(A, b, M=pc)` |
| `itl::pc::identity<...>`, `itl::pc::diagonal<...>` | the default, `ms.diagonal(A)` |
| `itl::bicgstab(A, x, b, pc, iter)` | `ms.bicgstab(A, b, M=pc)` |
| `basic_iteration` / `cyclic_iteration` / `noisy_iteration` | `full_output=True` for the history; `callback=` to watch each step |
| `iter.iterations()` | `report.iterations` |

The C++ programs store these tridiagonal matrices densely, as its own comments
note they shouldn't be in production. The Python version builds them sparse.
The heat example's header promises a comparison with the analytical solution
that the program doesn't make. This notebook makes it, and measures the
convergence rate.